<a href="https://colab.research.google.com/github/josenomberto/UTEC-CDIAV3-MCD8010/blob/main/Tarea_Actividad_Oversampling_Undersampling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Práctica: Submuestreo, sobremuestreo y manejo de desbalance de clases

**Curso:** Machine Learning — Posgrado UTEC  
**Tema:** Submuestreo aleatorio, sobremuestreo aleatorio, Tomek Links, SMOTE y su impacto en clasificadores  


## Integrantes del grupo (máximo 4)

| # | Nombre completo |
|---|---|
| 1 | |
| 2 | |
| 3 | |
| 4 | |

---

## Objetivo

Alcanzar un **F1-Score ≥ 0.65** sobre la **clase minoritaria (diabéticos)** en el conjunto de test, utilizando combinaciones de técnicas de remuestreo y clasificadores.

## Clasificadores requeridos

| Clasificador | Documentación |
|---|---|
| K-Nearest Neighbors | [sklearn KNeighborsClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsClassifier.html) |
| Regresión Logística | [sklearn LogisticRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html) |
| Support Vector Machine | [sklearn SVC](https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVC.html) |
| Naive Bayes | [sklearn GaussianNB](https://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.GaussianNB.html) |

## Técnicas de remuestreo requeridas

| Técnica | Tipo | Documentación |
|---|---|---|
| Submuestreo aleatorio | Submuestreo | [imblearn RandomUnderSampler](https://imbalanced-learn.org/stable/references/generated/imblearn.under_sampling.RandomUnderSampler.html) |
| Tomek Links | Submuestreo informado | [imblearn TomekLinks](https://imbalanced-learn.org/stable/references/generated/imblearn.under_sampling.TomekLinks.html) |
| Sobremuestreo aleatorio | Sobremuestreo | [imblearn RandomOverSampler](https://imbalanced-learn.org/stable/references/generated/imblearn.over_sampling.RandomOverSampler.html) |
| SMOTE | Sobremuestreo sintético | [imblearn SMOTE](https://imbalanced-learn.org/stable/references/generated/imblearn.over_sampling.SMOTE.html) |
| SMOTE + Tomek Links | Combinado | [imblearn SMOTETomek](https://imbalanced-learn.org/stable/references/generated/imblearn.combine.SMOTETomek.html) |
| Sin remuestreo | Baseline | — |

## Comparación de técnicas de muestreo

| Aspecto | Sub. aleatorio | Sobre. aleatorio |
|---|---|---|
| Información | Pierde datos | No pierde |
| Overfitting | Bajo | Alto |
| Datos nuevos | No | No (copias) |
| Complejidad | O(1) | O(1) |
| Tamaño final | Pequeño | Grande |
| Ruido | No agrega | Puede amplificar |

## ¿Cuándo usar cada técnica?

| Escenario | Recomendación |
|---|---|
| Dataset muy grande | Submuestreo aleatorio |
| Dataset pequeño | Sobremuestreo aleatorio |
| Frontera de decisión ruidosa | Submuestreo con Tomek Links |
| Muchas características | Submuestreo aleatorio |
| Desbalance moderado | Sobremuestreo aleatorio |
| Necesidad de interpretabilidad | Submuestreo (datos reales) |

## Pipeline correcto

```
Datos originales → Train/Test Split → Evaluar desbalance → [Muestreo SOLO en Train] → Entrenar → Evaluar con F1, AUC en Test original
```

**Regla de oro:** El muestreo se aplica **solo al conjunto de entrenamiento**. El conjunto de test debe mantener la distribución original para una evaluación realista.

## Reglas

1. El remuestreo se aplica **únicamente al conjunto de entrenamiento**
2. La evaluación se realiza sobre el **conjunto de test original** (sin remuestrear)
3. El F1-Score objetivo es sobre la **clase 1** (diabéticos)
4. Deben completar **todas** las celdas marcadas con `# TODO`

---
## Fase 0: Instalación de dependencias

Ejecutar esta celda para instalar las librerías necesarias.

- [`imbalanced-learn`](https://imbalanced-learn.org/stable/): Librería para técnicas de remuestreo
- [`scikit-learn`](https://scikit-learn.org/stable/): Clasificadores y métricas
- [`matplotlib`](https://matplotlib.org/): Visualización

In [ ]:
!pip install imbalanced-learn scikit-learn matplotlib pandas numpy -q

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from collections import Counter

# Métricas
# Docs: https://scikit-learn.org/stable/modules/model_evaluation.html
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)

# Preprocesamiento
# Docs: https://scikit-learn.org/stable/modules/preprocessing.html
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Clasificadores
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB

# Remuestreo
# Docs: https://imbalanced-learn.org/stable/references/index.html
from imblearn.over_sampling import SMOTE, RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler, TomekLinks
from imblearn.combine import SMOTETomek

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Umbral objetivo
F1_OBJETIVO = 0.65

print("Dependencias cargadas correctamente.")
print(f"F1-Score objetivo (clase minoritaria): {F1_OBJETIVO}")

Dependencias cargadas correctamente.
F1-Score objetivo (clase minoritaria): 0.65


---
## Fase 1: Descarga y exploración del dataset

### Dataset: Pima Indians Diabetes

Este dataset contiene información médica de **768 mujeres** de la comunidad Pima (Arizona, EE.UU.) para predecir si padecen **diabetes**.

| Feature | Descripción |
|---|---|
| `Pregnancies` | Número de embarazos |
| `Glucose` | Concentración de glucosa en plasma (mg/dL) |
| `BloodPressure` | Presión arterial diastólica (mm Hg) |
| `SkinThickness` | Espesor del pliegue cutáneo del tríceps (mm) |
| `Insulin` | Insulina sérica a 2 horas (mu U/mL) |
| `BMI` | Índice de masa corporal |
| `DiabetesPedigreeFunction` | Función de pedigrí de diabetes |
| `Age` | Edad (años) |
| **`Outcome`** | **0 = No diabética, 1 = Diabética** |

**Fuente:** [UCI Machine Learning Repository](https://www.kaggle.com/datasets/uciml/pima-indians-diabetes-database)

**Desbalance:** Aproximadamente **65% clase 0** (no diabética) vs **35% clase 1** (diabética)

In [ ]:
# ============================================================
# DESCARGA AUTOMÁTICA DEL DATASET
# ============================================================
# No modificar esta celda

url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
columnas = [
    'Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness',
    'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome'
]
df = pd.read_csv(url, header=None, names=columnas)

print(f"Dataset descargado: {df.shape[0]} filas × {df.shape[1]} columnas")
print(f"\nDistribución de la variable objetivo (Outcome):")
print(df['Outcome'].value_counts())
print(f"\nPorcentaje de diabéticas (clase 1): {df['Outcome'].mean()*100:.1f}%")
df.head()

Dataset descargado: 768 filas × 9 columnas

Distribución de la variable objetivo (Outcome):
Outcome
0    500
1    268
Name: count, dtype: int64

Porcentaje de diabéticas (clase 1): 34.9%


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


### 1.1 Exploración básica

**Guía:** Analicen las estadísticas descriptivas del dataset. Observen si hay valores sospechosos (e.g., `Glucose=0`, `BloodPressure=0`) que podrían ser datos faltantes codificados como cero.

Recurso útil: [pandas `describe()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.describe.html)

In [ ]:
# TODO: Mostrar estadísticas descriptivas del dataset
# Pista: usar df.describe()


### 1.2 Visualización del desbalance

**Guía:** Crear un gráfico de barras que muestre la distribución de la clase objetivo. Esto les ayudará a dimensionar visualmente el desbalance.

Recurso útil: [matplotlib `bar()`](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.bar.html)

In [ ]:
# TODO: Crear un gráfico de barras con la distribución de clases
# Pista: usar df['Outcome'].value_counts().plot(kind='bar', ...)
# Usar colores distintos para cada clase


---
## Fase 2: Preprocesamiento

**Guía:**
1. Separar features (`X`) y target (`y`)
2. Dividir en train/test con `test_size=0.3` y `stratify=y`
3. Escalar las features con `StandardScaler` (fit solo en train, transform en ambos)

**Importante:** El escalado se ajusta **solo** con los datos de entrenamiento para evitar data leakage.

Recursos útiles:
- [train_test_split](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html)
- [StandardScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html)

In [ ]:
# TODO: Separar features y target
# X = ...
# y = ...


# TODO: Dividir en train/test (70/30), estratificado
# X_train, X_test, y_train, y_test = train_test_split(...)


# TODO: Escalar features (fit en train, transform en ambos)
# scaler = StandardScaler()
# X_train_scaled = ...
# X_test_scaled = ...


# Verificar distribución
# print(f"Train: {Counter(y_train)}")
# print(f"Test:  {Counter(y_test)}")

---
## Fase 3: Baseline (sin remuestreo)

**Guía:** Entrenar los 4 clasificadores **sin ningún remuestreo** y registrar sus métricas. Este será el punto de referencia contra el cual medirán las mejoras.

**Pregunta para reflexionar:** ¿Qué métrica es más relevante en un problema de diagnóstico médico: accuracy o recall? ¿Por qué?

Recursos útiles:
- [classification_report](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.classification_report.html)
- [f1_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.f1_score.html)

In [ ]:
# Función auxiliar para evaluar un clasificador (NO MODIFICAR)
def evaluar(nombre, y_true, y_pred):
    """Evalúa y retorna las métricas de un clasificador."""
    return {
        'Modelo': nombre,
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision (1)': precision_score(y_true, y_pred),
        'Recall (1)': recall_score(y_true, y_pred),
        'F1 (1)': f1_score(y_true, y_pred)
    }

In [ ]:
# TODO: Entrenar los 4 clasificadores SIN remuestreo y recopilar métricas
#
# clasificadores = {
#     'KNN':       KNeighborsClassifier(...),
#     'Logística':  LogisticRegression(max_iter=1000, random_state=42),
#     'SVM':       SVC(random_state=42),
#     'Naive Bayes': GaussianNB()
# }
#
# resultados_baseline = []
# for nombre, clf in clasificadores.items():
#     clf.fit(X_train_scaled, y_train)
#     y_pred = clf.predict(X_test_scaled)
#     resultados_baseline.append(evaluar(f'{nombre} (baseline)', y_test, y_pred))
#
# df_baseline = pd.DataFrame(resultados_baseline).round(4)
# df_baseline


### 3.1 Reflexión sobre el baseline

**Preguntas guía (responder en la celda siguiente):**

1. ¿Cuál clasificador tiene el mejor F1 sobre la clase 1 sin remuestreo?
2. ¿Alguno supera el objetivo de F1 ≥ 0.65? Si no, ¿qué tan lejos están?
3. ¿Observan diferencia entre accuracy y F1? ¿A qué se debe?
4. ¿Qué significa un recall bajo en el contexto de diagnóstico de diabetes?

**Respuestas:**

1. ...
2. ...
3. ...
4. ...

---
## Fase 4: Experimentación con técnicas de submuestreo

El submuestreo reduce la clase mayoritaria para equilibrar las clases. Veremos dos variantes:
- **Submuestreo aleatorio:** elimina muestras de la clase mayoritaria al azar
- **Tomek Links:** elimina selectivamente pares de puntos de clases opuestas que son mutuamente vecinos más cercanos, limpiando la frontera de decisión

### 4.1 Submuestreo aleatorio

**Guía:** El submuestreo aleatorio **elimina** muestras de la clase mayoritaria hasta igualar la minoritaria. La desventaja es la pérdida de información potencialmente útil.

**Pregunta para reflexionar:** ¿Qué pasa con la cantidad total de datos de entrenamiento tras el submuestreo? ¿Cómo podría afectar esto al rendimiento?

Recurso útil: [RandomUnderSampler](https://imbalanced-learn.org/stable/references/generated/imblearn.under_sampling.RandomUnderSampler.html)

In [ ]:
# TODO: Aplicar submuestreo aleatorio al conjunto de entrenamiento
#
# rus = RandomUnderSampler(random_state=42)
# X_train_rus, y_train_rus = rus.fit_resample(X_train_scaled, y_train)
#
# print(f"Antes:   {Counter(y_train)} → Total: {len(y_train)}")
# print(f"Después: {Counter(y_train_rus)} → Total: {len(y_train_rus)}")


In [ ]:
# TODO: Entrenar los 4 clasificadores con datos submuestreados
# Evaluar en X_test_scaled (sin remuestrear)
#
# resultados_rus = []
# for nombre, clf in clasificadores.items():
#     clf.fit(X_train_rus, y_train_rus)
#     y_pred = clf.predict(X_test_scaled)
#     resultados_rus.append(evaluar(f'{nombre} + Sub. aleatorio', y_test, y_pred))
#
# df_rus = pd.DataFrame(resultados_rus).round(4)
# df_rus


### 4.2 Submuestreo con Tomek Links

**Guía:** Dos puntos $(\mathbf{x}_i, \mathbf{x}_j)$ de clases distintas forman un **Tomek Link** si no existe un tercer punto $\mathbf{x}_k$ tal que:

$$d(\mathbf{x}_i, \mathbf{x}_k) < d(\mathbf{x}_i, \mathbf{x}_j) \quad \text{o} \quad d(\mathbf{x}_j, \mathbf{x}_k) < d(\mathbf{x}_i, \mathbf{x}_j)$$

Al eliminar el punto de la clase mayoritaria en cada Tomek Link, se limpia la frontera de decisión sin eliminar datos aleatoriamente.

Recurso útil: [TomekLinks](https://imbalanced-learn.org/stable/references/generated/imblearn.under_sampling.TomekLinks.html)

In [ ]:
# TODO: Aplicar Tomek Links al conjunto de entrenamiento
#
# tomek = TomekLinks()
# X_train_tomek, y_train_tomek = tomek.fit_resample(X_train_scaled, y_train)
#
# print(f"Antes:   {Counter(y_train)} → Total: {len(y_train)}")
# print(f"Después: {Counter(y_train_tomek)} → Total: {len(y_train_tomek)}")
# print(f"Tomek Links eliminados: {len(y_train) - len(y_train_tomek)}")


In [ ]:
# TODO: Entrenar los 4 clasificadores con datos limpiados por Tomek Links
# Evaluar en X_test_scaled (sin remuestrear)
#
# resultados_tomek = []
# for nombre, clf in clasificadores.items():
#     clf.fit(X_train_tomek, y_train_tomek)
#     y_pred = clf.predict(X_test_scaled)
#     resultados_tomek.append(evaluar(f'{nombre} + Tomek Links', y_test, y_pred))
#
# df_tomek = pd.DataFrame(resultados_tomek).round(4)
# df_tomek


---
## Fase 5: Experimentación con técnicas de sobremuestreo

El sobremuestreo incrementa la clase minoritaria para equilibrar las clases. Veremos dos variantes:
- **Sobremuestreo aleatorio:** duplica muestras existentes de la clase minoritaria
- **SMOTE:** genera muestras *sintéticas* por interpolación entre puntos minoritarios y sus K vecinos más cercanos

### 5.1 Sobremuestreo aleatorio

**Guía:** El sobremuestreo aleatorio simplemente **duplica** muestras existentes de la clase minoritaria. No agrega información nueva y puede causar sobreajuste, ya que el modelo memoriza los pocos ejemplos minoritarios.

Recurso útil: [RandomOverSampler](https://imbalanced-learn.org/stable/references/generated/imblearn.over_sampling.RandomOverSampler.html)

In [ ]:
# TODO: Aplicar sobremuestreo aleatorio al conjunto de entrenamiento
#
# ros = RandomOverSampler(random_state=42)
# X_train_ros, y_train_ros = ros.fit_resample(X_train_scaled, y_train)
#
# print(f"Antes:   {Counter(y_train)} → Total: {len(y_train)}")
# print(f"Después: {Counter(y_train_ros)} → Total: {len(y_train_ros)}")


In [ ]:
# TODO: Entrenar los 4 clasificadores con datos sobremuestreados
# Evaluar en X_test_scaled (sin remuestrear)
#
# resultados_ros = []
# for nombre, clf in clasificadores.items():
#     clf.fit(X_train_ros, y_train_ros)
#     y_pred = clf.predict(X_test_scaled)
#     resultados_ros.append(evaluar(f'{nombre} + Sobre. aleatorio', y_test, y_pred))
#
# df_ros = pd.DataFrame(resultados_ros).round(4)
# df_ros


### 5.2 Sobremuestreo con SMOTE

**Guía:** SMOTE genera muestras sintéticas interpolando entre puntos minoritarios y sus K vecinos más cercanos:

$$\mathbf{x}_{\text{new}} = \mathbf{x}_i + \lambda \cdot (\hat{\mathbf{x}}_i - \mathbf{x}_i), \quad \lambda \sim U(0,1)$$

A diferencia del sobremuestreo aleatorio, SMOTE **genera datos nuevos** que no son copias exactas, lo que reduce el riesgo de overfitting.

Parámetro clave: `k_neighbors` (por defecto 5). Pueden experimentar con valores entre 3 y 7.

Recurso útil: [SMOTE](https://imbalanced-learn.org/stable/references/generated/imblearn.over_sampling.SMOTE.html)

In [ ]:
# TODO: Aplicar SMOTE al conjunto de entrenamiento
#
# smote = SMOTE(k_neighbors=5, random_state=42)
# X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)
#
# print(f"Antes:   {Counter(y_train)}")
# print(f"Después: {Counter(y_train_smote)}")


In [ ]:
# TODO: Entrenar los 4 clasificadores con datos remuestreados por SMOTE
# Evaluar en X_test_scaled (sin remuestrear)
#
# resultados_smote = []
# for nombre, clf in clasificadores.items():
#     clf.fit(X_train_smote, y_train_smote)
#     y_pred = clf.predict(X_test_scaled)
#     resultados_smote.append(evaluar(f'{nombre} + SMOTE', y_test, y_pred))
#
# df_smote = pd.DataFrame(resultados_smote).round(4)
# df_smote


---
## Fase 6: Técnica combinada (SMOTE + Tomek Links)

**Guía:** Esta técnica primero aplica SMOTE (sobremuestreo sintético) y luego elimina los **Tomek Links** — pares de puntos de clases opuestas que son mutuamente vecinos más cercanos. Esto combina lo mejor de ambos enfoques: genera datos sintéticos y luego "limpia" la frontera de decisión.

Recurso útil: [SMOTETomek](https://imbalanced-learn.org/stable/references/generated/imblearn.combine.SMOTETomek.html)

In [ ]:
# TODO: Aplicar SMOTE + Tomek Links y entrenar los 4 clasificadores
#
# smt = SMOTETomek(random_state=42)
# X_train_smt, y_train_smt = smt.fit_resample(X_train_scaled, y_train)
#
# print(f"Antes:   {Counter(y_train)} → Total: {len(y_train)}")
# print(f"Después: {Counter(y_train_smt)} → Total: {len(y_train_smt)}")
#
# resultados_smt = []
# for nombre, clf in clasificadores.items():
#     clf.fit(X_train_smt, y_train_smt)
#     y_pred = clf.predict(X_test_scaled)
#     resultados_smt.append(evaluar(f'{nombre} + SMOTE+Tomek', y_test, y_pred))
#
# df_smt = pd.DataFrame(resultados_smt).round(4)
# df_smt


---
## Fase 7: Tabla comparativa completa

**Guía:** Consoliden todos los resultados en una sola tabla para facilitar la comparación. Identifiquen qué combinación (remuestreo + clasificador) alcanza el F1 objetivo.

Recurso útil: [pandas `concat()`](https://pandas.pydata.org/docs/reference/api/pandas.concat.html), [`style.apply()`](https://pandas.pydata.org/docs/reference/api/pandas.io.formats.style.Styler.apply.html)

In [ ]:
# TODO: Consolidar todos los resultados en un único DataFrame
#
# df_todos = pd.concat([
#     df_baseline,
#     df_rus,
#     df_tomek,
#     df_ros,
#     df_smote,
#     df_smt
# ], ignore_index=True)
#
# # Marcar cuáles superan el objetivo
# df_todos['¿Supera F1 ≥ 0.65?'] = df_todos['F1 (1)'] >= F1_OBJETIVO
# df_todos.sort_values('F1 (1)', ascending=False)


---
## Fase 8: Visualización de resultados

**Guía:** Crear visualizaciones que permitan comparar las métricas entre configuraciones. Se sugieren:

1. **Gráfico de barras agrupadas** con el F1-Score de cada combinación
2. **Matrices de confusión** de la mejor y peor configuración
3. **Heatmap** con todas las métricas

Recursos útiles:
- [matplotlib barras agrupadas](https://matplotlib.org/stable/gallery/lines_bars_and_markers/barchart.html)
- [ConfusionMatrixDisplay](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.ConfusionMatrixDisplay.html)
- [matplotlib `imshow()` para heatmaps](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.imshow.html)

### 8.1 F1-Score por combinación

**Guía:** Crear un gráfico de barras horizontales ordenado por F1-Score. Dibujar una línea vertical en F1 = 0.65 para marcar el objetivo.

In [ ]:
# TODO: Crear gráfico de barras horizontales con F1-Score
#
# Pista:
# df_sorted = df_todos.sort_values('F1 (1)')
# fig, ax = plt.subplots(figsize=(10, 10))
# colors = ['#4ECDC4' if v >= F1_OBJETIVO else '#FF6B6B' for v in df_sorted['F1 (1)']]
# ax.barh(df_sorted['Modelo'], df_sorted['F1 (1)'], color=colors, edgecolor='black')
# ax.axvline(x=F1_OBJETIVO, color='red', linestyle='--', label=f'Objetivo F1={F1_OBJETIVO}')
# ax.legend()
# ...


### 8.2 Matrices de confusión

**Guía:** Comparar la matriz de confusión del **mejor baseline** vs la **mejor configuración con remuestreo**. Observar cómo cambian los falsos negativos (diabéticas no detectadas).

In [ ]:
# TODO: Mostrar matrices de confusión comparativas
#
# Pista: Reentrenar el mejor baseline y la mejor config con remuestreo,
# obtener predicciones, y usar ConfusionMatrixDisplay
#
# fig, axes = plt.subplots(1, 2, figsize=(12, 5))
# ConfusionMatrixDisplay.from_predictions(y_test, y_pred_mejor_baseline,
#     display_labels=['No diab.', 'Diab.'], ax=axes[0], cmap='Blues')
# ...


### 8.3 Heatmap comparativo (opcional)

**Guía:** Crear un heatmap donde las filas sean las combinaciones y las columnas las métricas.

In [ ]:
# TODO (opcional): Crear un heatmap con todas las métricas
#
# Pista:
# data = df_todos.set_index('Modelo')[['Accuracy', 'Precision (1)', 'Recall (1)', 'F1 (1)']]
# fig, ax = plt.subplots(figsize=(10, 10))
# im = ax.imshow(data.values, cmap='YlOrRd', aspect='auto')
# ...


---
## Fase 9: Impacto del muestreo según el clasificador

El efecto del muestreo **no es igual para todos los clasificadores**. Analicen la siguiente tabla y contrasten con sus resultados experimentales:

| Clasificador | Efecto positivo del muestreo | Efecto negativo del muestreo |
|---|---|---|
| **Regresión logística** | Mejora recall cuando no se dispone de `class_weight` | Distorsiona las probabilidades calibradas; requiere recalibración |
| **SVM** | Submuestreo reduce el costo O(N²–N³); mejora la frontera | Sobremuestreo duplica vectores de soporte idénticos sin beneficio |
| **k-NN** | Submuestreo reduce el ruido de la clase mayoritaria en la vecindad | Sobremuestreo crea vecindarios densos artificiales; sesga las votaciones |
| **Naive Bayes** | Puede mejorar con desbalance extremo | Altera las frecuencias a priori P(Y=k); las estimaciones se sesgan |

---
## Fase 10: Análisis y discusión

Responder las siguientes preguntas con base en los resultados obtenidos. Cada respuesta debe estar fundamentada con evidencia de los experimentos.

### Preguntas obligatorias

**P1: ¿Qué combinación (remuestreo + clasificador) obtuvo el mejor F1 sobre la clase diabética? ¿Por qué creen que fue la mejor?**

*Respuesta:*


**P2: Comparen el efecto del submuestreo (aleatorio, Tomek Links) vs el sobremuestreo (aleatorio, SMOTE). ¿Cuál funcionó mejor en general? ¿Qué fortalezas y debilidades observaron en cada enfoque?**

*Pista: Consideren la cantidad de datos de entrenamiento resultante, la pérdida de información, y el efecto sobre cada métrica.*

*Respuesta:*


**P3: ¿Hubo algún clasificador que se benefició más del remuestreo? ¿Alguno que empeoró? Expliquen con base en la tabla de la Fase 9.**

*Pista: Piensen en cómo cada clasificador construye su frontera de decisión:*
- *KNN: basado en vecindad local → el submuestreo reduce el ruido de la clase mayoritaria en la vecindad*
- *Regresión Logística: estima P(Y|X) directamente → el muestreo puede distorsionar las probabilidades*
- *SVM: maximiza el margen → el sobremuestreo duplica vectores de soporte sin beneficio*
- *Naive Bayes: basado en probabilidades a priori → el muestreo altera P(Y=k)*

*Respuesta:*


**P4: ¿Qué diferencia observaron entre SMOTE y el sobremuestreo aleatorio? ¿Y entre el submuestreo aleatorio y Tomek Links?**


*Respuesta:*


**P5: ¿Qué impacto tuvo el remuestreo sobre el accuracy? ¿Por qué el accuracy puede ser engañoso en datasets desbalanceados?**

*Respuesta:*


**P6: En el contexto médico de este dataset, ¿qué es más grave: un falso positivo (diagnosticar diabetes a alguien sano) o un falso negativo (no detectar diabetes)? ¿Cómo afecta esto a la elección de métrica y de técnica de muestreo?**

*Respuesta:*


## Fase 11: Conclusiones

**Guía:** Redacten al menos 4 conclusiones que sinteticen lo aprendido en esta práctica. Relacionen sus hallazgos con los conceptos teóricos vistos en las sesiones de submuestreo, sobremuestreo y SMOTE.

**Conclusiones:**

1. ...
2. ...
3. ...
4. ...